In [0]:
from pyspark.sql.functions import * #col,trim
from pyspark.sql.types import * #StringType

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Data Tranformations

In [0]:
display(df)

#### 1. Trim the extra spaces


In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#### 2. Standardization for marital_status and gender column

In [0]:
df = df.withColumn('cst_marital_status', 
                   when(col('cst_marital_status') == 'S', 'Single')
                   .when(col('cst_marital_status') == 'M', 'Married')
                   .otherwise(col('cst_marital_status'))
        ).withColumn('cst_gndr', 
                   when(col('cst_gndr') == 'M', 'Male')
                   .when(col('cst_gndr') == 'F', 'Female')
                   .otherwise(col('cst_gndr'))
        )

#### 3, Remove records with missing Customer ID

In [0]:
df =  df.dropna(subset=['cst_id'])

#### 4. Column names and some table names are not friendly

In [0]:
RENAME_MAP = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'customer_firstname',
    'cst_lastname': 'customer_lastname',
    'cst_marital_status': 'customer_marital_status',
    'cst_gndr': 'customer_gender',
    'cst_create_date': 'customer_create_date'
}

In [0]:
for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

In [0]:
display(df)

#Write Into Silver Table

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.crm_customers')

In [0]:
%sql
select * from workspace.silver.crm_customers limit 5